# Prepare NSW Permanent Hourly Traffic Data for Scenario C

This notebook processes the Transport for NSW permanent traffic-count files.

## What the uploaded sample contains

The hourly file is in **wide daily format**:

- one row for a traffic station, direction, vehicle classification, and date
- `hour_00` to `hour_23` contain the hourly counts
- `station_key` links the count data to the station-reference file

## What this notebook produces

1. Scans every file named `road_traffic_counts_hourly_permanent*.csv`
2. Filters the study period to **2019–2024**
3. Filters to selected permanent traffic stations
4. Converts `hour_00 ... hour_23` into one row per hour
5. Sums directions/classifications into total hourly traffic
6. Creates coverage and missingness summaries
7. Saves a clean table for future Scenario C merging

## Important

Only one hourly chunk was uploaded in this chat.  
Place all five extracted CSV chunks in the same folder before running this notebook locally.


## 1. Imports and folders

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(".")
RAW_TRAFFIC_DIR = BASE_DIR / "data" / "context" / "traffic" / "raw"
PROCESSED_TRAFFIC_DIR = BASE_DIR / "data" / "context" / "traffic" / "processed"
PROCESSED_TRAFFIC_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_FILE = RAW_TRAFFIC_DIR / "road_traffic_counts_station_reference.csv"

# All extracted permanent hourly files should match this pattern.
HOURLY_PATTERN = "road_traffic_counts_hourly_permanent*.csv"

START_YEAR = 2019
END_YEAR = 2024

print("Raw folder:", RAW_TRAFFIC_DIR.resolve())
print("Processed folder:", PROCESSED_TRAFFIC_DIR.resolve())


c:\Users\MSI\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\MSI\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Raw folder: C:\Users\MSI\Desktop\Research Perp\Code\data\context\traffic\raw
Processed folder: C:\Users\MSI\Desktop\Research Perp\Code\data\context\traffic\processed


## 2. Find all traffic files

The ZIP is split into several large CSV files.  
The code processes them in chunks so it does not load several gigabytes into memory at once.


In [2]:
hourly_files = sorted(RAW_TRAFFIC_DIR.glob(HOURLY_PATTERN))

print("Hourly files found:", len(hourly_files))
for file in hourly_files:
    print("-", file.name, f"({file.stat().st_size / 1024**2:.1f} MB)")

if not hourly_files:
    raise FileNotFoundError(
        f"No files matching {HOURLY_PATTERN!r} were found in {RAW_TRAFFIC_DIR.resolve()}"
    )

if not REFERENCE_FILE.exists():
    raise FileNotFoundError(f"Missing station reference file: {REFERENCE_FILE.resolve()}")


Hourly files found: 5
- road_traffic_counts_hourly_permanent0.csv (195.5 MB)
- road_traffic_counts_hourly_permanent1.csv (199.8 MB)
- road_traffic_counts_hourly_permanent2.csv (201.1 MB)
- road_traffic_counts_hourly_permanent3.csv (204.7 MB)
- road_traffic_counts_hourly_permanent4.csv (24.0 MB)


## 3. Load and inspect station reference

In [3]:
reference = pd.read_csv(REFERENCE_FILE)

reference["station_key"] = pd.to_numeric(
    reference["station_key"], errors="coerce"
).astype("Int64")

reference["permanent_station"] = pd.to_numeric(
    reference["permanent_station"], errors="coerce"
).fillna(0).astype(int)

permanent_reference = reference[
    (reference["permanent_station"] == 1) &
    (reference["station_key"].notna())
].copy()

reference_columns = [
    "station_key",
    "station_id",
    "full_name",
    "road_name",
    "suburb",
    "quality_rating",
    "wgs84_latitude",
    "wgs84_longitude",
]

permanent_reference[reference_columns].head()


,station_key,station_id,full_name,road_name,suburb,quality_rating,wgs84_latitude,wgs84_longitude
0,15828002,7211,"Lily Lane, South of Fern Circuit",Lily Lane,Adamstown,5,-32.940571,151.713120
6,57257,66249,"Fairfield Street, East of Cockburn Crescent",Fairfield Street,Fairfield East,4,-33.869324,150.963486
11,15828011,T0281,"Hume Highway, East of George Street",Hume Highway Hvcs,Marulan,5,-34.703720,150.016647
12,15252004,6154,"New England Highway, South of Muscle Creek Road",New England Highway,Muswellbrook,4,-32.300934,150.932449
15,57052,50250,"Hart Drive, East of Mahony Road",Hart Drive,Constitution Hill,4,-33.797592,150.975433


## 4. Produce candidate traffic stations

This searches by suburb first.

Suburb matching is only an initial filter. For the final research dataset, choose a counter based on:

- distance from the air-quality monitor
- permanent-station status
- traffic-data coverage during 2019–2024
- data-quality rating
- road relevance

The uploaded sample contained permanent counters for Rozelle, Liverpool, and St Marys.  
Randwick, Bringelly, and Richmond may be present in another CSV chunk or under a nearby suburb name.


In [4]:
AIR_QUALITY_AREAS = {
    "ROZELLE": ["Rozelle"],
    "RANDWICK": ["Randwick", "Kingsford", "Kensington"],
    "LIVERPOOL": ["Liverpool"],
    "ST_MARYS": ["St Marys", "North St Marys"],
    "BRINGELLY": ["Bringelly", "Rossmore", "Leppington"],
    "RICHMOND": ["Richmond", "Windsor", "Hobartville"],
}

candidate_frames = []

for air_station, suburb_terms in AIR_QUALITY_AREAS.items():
    mask = pd.Series(False, index=permanent_reference.index)

    for term in suburb_terms:
        mask |= permanent_reference["suburb"].astype(str).str.contains(
            term, case=False, na=False
        )

    candidates = permanent_reference.loc[mask, reference_columns].copy()
    candidates.insert(0, "air_station", air_station)
    candidate_frames.append(candidates)

candidate_reference = pd.concat(candidate_frames, ignore_index=True)
candidate_reference.to_csv(
    PROCESSED_TRAFFIC_DIR / "traffic_station_candidates_by_area.csv",
    index=False,
)

candidate_reference


,air_station,station_key,station_id,full_name,road_name,suburb,quality_rating,wgs84_latitude,wgs84_longitude
0,ROZELLE,56705,20001,"Western Distributor, West of Bowman Street",Western Distributor,Rozelle,5,-33.869068,151.185349
1,ROZELLE,56846,30011,"Victoria Road, North of Byrnes Street",Victoria Road,Rozelle,4,-33.860798,151.164291
2,ROZELLE,15286012,7163,"Victoria Road, North of Evans Street",Victoria Road,Rozelle,5,-33.865181,151.171997
3,LIVERPOOL,57182,60045,"Copeland Street, North of Campbell Street",Copeland Street,Liverpool,4,-33.916756,150.919952
4,LIVERPOOL,57183,60046,"Orange Grove Road, West of Links Avenue",Orange Grove Road,Liverpool,4,-33.904961,150.917831
5,LIVERPOOL,57176,60005,"Hume Highway, North of South Western Motorway",Hume Highway,Liverpool,5,-33.937553,150.913849
6,LIVERPOOL,57178,60032,"Elizabeth Drive, South of Reservoir Road",Elizabeth Drive,Liverpool,4,-33.914284,150.907883
7,ST_MARYS,99990004,7123-PR,"Pages Road, North of Great Western Highway",Pages Road,St Marys,5,-33.769588,150.768738
8,BRINGELLY,57503,85019,"Camden Valley Way, West of Heath Road",Camden Valley Way,Leppington,4,-33.976528,150.808868
9,RICHMOND,57555,88046,"Bridge Street, North of Court Street",Bridge Street,Windsor,4,-33.605782,150.824585


## 5. Optional: select nearest counters using air-monitor coordinates

For a defensible final selection, add the official latitude and longitude of each air-quality station below.

The notebook will calculate straight-line distance using the Haversine formula.

Leave coordinates as `None` until they are verified from an official NSW source.


In [5]:
AIR_MONITOR_COORDS = {
    "ROZELLE": None,      # Example format: (-33.0000, 151.0000)
    "RANDWICK": None,
    "LIVERPOOL": None,
    "ST_MARYS": None,
    "BRINGELLY": None,
    "RICHMOND": None,
}

def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0088

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 2 * radius_km * np.arcsin(np.sqrt(a))

nearest_frames = []

for air_station, coords in AIR_MONITOR_COORDS.items():
    if coords is None:
        continue

    lat, lon = coords
    temp = permanent_reference[reference_columns].copy()

    temp["distance_km"] = haversine_km(
        lat,
        lon,
        pd.to_numeric(temp["wgs84_latitude"], errors="coerce"),
        pd.to_numeric(temp["wgs84_longitude"], errors="coerce"),
    )

    temp.insert(0, "air_station", air_station)
    nearest_frames.append(temp.nsmallest(10, "distance_km"))

if nearest_frames:
    nearest_candidates = pd.concat(nearest_frames, ignore_index=True)
    nearest_candidates.to_csv(
        PROCESSED_TRAFFIC_DIR / "nearest_traffic_station_candidates.csv",
        index=False,
    )
    display(nearest_candidates)
else:
    print("Coordinates have not been entered yet. Area-based candidates were created instead.")


Coordinates have not been entered yet. Area-based candidates were created instead.


## 6. Select traffic station keys

After checking candidate locations and data coverage, put the chosen traffic `station_key` values here.

You may select:

- one counter per air-quality station for simple interpretation, or
- several nearby counters and aggregate them into an area total

The values below are examples observed in the uploaded reference/sample and must still be confirmed against full 2019–2024 coverage.


In [6]:
SELECTED_STATION_KEYS = {
    # Rozelle permanent counters observed in the uploaded reference/sample:
    "ROZELLE": [56705, 56846, 15286012],

    # Liverpool permanent counters observed in the uploaded reference/sample:
    "LIVERPOOL": [57182, 57183, 57176, 57178],

    # St Marys permanent counter observed in the uploaded reference/sample:
    "ST_MARYS": [99990004],

    # Add verified station keys after scanning every hourly file:
    "RANDWICK": [],
    "BRINGELLY": [],
    "RICHMOND": [],
}

selected_key_to_area = {
    int(station_key): area
    for area, station_keys in SELECTED_STATION_KEYS.items()
    for station_key in station_keys
}

selected_keys = sorted(selected_key_to_area)

print("Selected station keys:", selected_keys)

if not selected_keys:
    raise ValueError("No traffic station keys have been selected.")


Selected station keys: [56705, 56846, 57176, 57178, 57182, 57183, 15286012, 99990004]


## 7. Scan all hourly CSV files

Only the required columns are read.

The file stores one date with 24 separate hourly columns.  
Filtering happens before reshaping, which saves memory.


In [7]:
HOUR_COLUMNS = [f"hour_{hour:02d}" for hour in range(24)]

USE_COLUMNS = [
    "station_key",
    "traffic_direction_seq",
    "cardinal_direction_seq",
    "classification_seq",
    "date",
    "year",
    "daily_total",
    *HOUR_COLUMNS,
]

filtered_daily_parts = []
file_scan_rows = []

for file_path in hourly_files:
    file_total_rows = 0
    file_selected_rows = 0
    file_min_date = None
    file_max_date = None

    for chunk in pd.read_csv(
        file_path,
        usecols=USE_COLUMNS,
        chunksize=250_000,
        low_memory=False,
    ):
        file_total_rows += len(chunk)

        chunk["station_key"] = pd.to_numeric(
            chunk["station_key"], errors="coerce"
        ).astype("Int64")

        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")

        selected = chunk[
            chunk["station_key"].isin(selected_keys)
            & chunk["year"].between(START_YEAR, END_YEAR)
        ].copy()

        if selected.empty:
            continue

        selected["date"] = pd.to_datetime(
            selected["date"], errors="coerce", utc=True
        ).dt.tz_convert(None).dt.normalize()

        selected = selected.dropna(subset=["date", "station_key"])

        if selected.empty:
            continue

        file_selected_rows += len(selected)
        current_min = selected["date"].min()
        current_max = selected["date"].max()

        file_min_date = (
            current_min
            if file_min_date is None
            else min(file_min_date, current_min)
        )
        file_max_date = (
            current_max
            if file_max_date is None
            else max(file_max_date, current_max)
        )

        filtered_daily_parts.append(selected)

    file_scan_rows.append(
        {
            "file": file_path.name,
            "rows_scanned": file_total_rows,
            "selected_daily_rows": file_selected_rows,
            "selected_min_date": file_min_date,
            "selected_max_date": file_max_date,
        }
    )

file_scan_summary = pd.DataFrame(file_scan_rows)
file_scan_summary.to_csv(
    PROCESSED_TRAFFIC_DIR / "traffic_file_scan_summary.csv",
    index=False,
)

file_scan_summary


,file,rows_scanned,selected_daily_rows,selected_min_date,selected_max_date
0,road_traffic_counts_hourly_permanent0.csv,1000000,114,2020-04-01,2020-04-30
1,road_traffic_counts_hourly_permanent1.csv,1000000,3518,2020-04-01,2021-03-31
2,road_traffic_counts_hourly_permanent2.csv,1000000,1554,2021-03-01,2022-02-14
3,road_traffic_counts_hourly_permanent3.csv,1000000,10529,2019-01-01,2024-12-31
4,road_traffic_counts_hourly_permanent4.csv,117831,0,NaT,NaT


## 8. Combine filtered daily records

In [8]:
if not filtered_daily_parts:
    raise ValueError(
        "No selected station records were found for 2019–2024. "
        "Check station keys and scan all extracted CSV chunks."
    )

traffic_daily = pd.concat(filtered_daily_parts, ignore_index=True)

traffic_daily = traffic_daily.drop_duplicates(
    subset=[
        "station_key",
        "date",
        "traffic_direction_seq",
        "cardinal_direction_seq",
        "classification_seq",
    ]
).reset_index(drop=True)

print("Filtered daily-direction/classification rows:", len(traffic_daily))
print("Date range:", traffic_daily["date"].min(), "to", traffic_daily["date"].max())
print("Station keys found:", sorted(traffic_daily["station_key"].dropna().astype(int).unique()))


Filtered daily-direction/classification rows: 15715
Date range: 2019-01-01 00:00:00 to 2024-12-31 00:00:00
Station keys found: [56705, 56846, 57176, 57178, 15286012, 99990004]


## 9. Convert daily wide data into hourly long data

Example transformation:

`date | hour_00 | hour_01 | ...`

becomes:

`DATETIME | TRAFFIC_COUNT`


In [9]:
traffic_long = traffic_daily.melt(
    id_vars=[
        "station_key",
        "date",
        "traffic_direction_seq",
        "cardinal_direction_seq",
        "classification_seq",
    ],
    value_vars=HOUR_COLUMNS,
    var_name="hour_column",
    value_name="traffic_count",
)

traffic_long["hour"] = (
    traffic_long["hour_column"]
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

traffic_long["DATETIME"] = (
    traffic_long["date"]
    + pd.to_timedelta(traffic_long["hour"], unit="h")
)

traffic_long["traffic_count"] = pd.to_numeric(
    traffic_long["traffic_count"], errors="coerce"
)

# Negative counts are invalid and become missing.
traffic_long.loc[
    traffic_long["traffic_count"] < 0, "traffic_count"
] = np.nan

traffic_long.head()


,station_key,date,traffic_direction_seq,cardinal_direction_seq,classification_seq,hour_column,traffic_count,hour,DATETIME
0,99990004,2020-04-24,0,3,3,hour_00,2.0,0,2020-04-24
1,56705,2020-04-29,1,3,0,hour_00,168.0,0,2020-04-29
2,57178,2020-04-08,0,3,0,hour_00,97.0,0,2020-04-08
3,99990004,2020-04-06,0,3,2,hour_00,44.0,0,2020-04-06
4,99990004,2020-04-07,1,7,2,hour_00,166.0,0,2020-04-07


## 10. Aggregate directions and vehicle classes

Several records may exist for the same station and hour because of:

- traffic direction
- carriageway direction
- vehicle classification

For the main Scenario C traffic-volume feature, the code sums all available records into total hourly traffic.

It also saves the number of contributing source records for quality checking.


In [10]:
traffic_station_hourly = (
    traffic_long
    .groupby(["station_key", "DATETIME"], as_index=False)
    .agg(
        traffic_count=("traffic_count", lambda values: values.sum(min_count=1)),
        contributing_records=("traffic_count", "count"),
    )
)

traffic_station_hourly["area"] = (
    traffic_station_hourly["station_key"]
    .astype(int)
    .map(selected_key_to_area)
)

traffic_station_hourly = traffic_station_hourly.merge(
    permanent_reference[reference_columns],
    on="station_key",
    how="left",
)

traffic_station_hourly.to_csv(
    PROCESSED_TRAFFIC_DIR / "traffic_hourly_by_counter.csv",
    index=False,
)

traffic_station_hourly.head()


,station_key,DATETIME,traffic_count,contributing_records,area,station_id,full_name,road_name,suburb,quality_rating,wgs84_latitude,wgs84_longitude
0,56705,2019-01-01 00:00:00,1655.0,1,ROZELLE,20001,"Western Distributor, West of Bowman Street",Western Distributor,Rozelle,5,-33.869068,151.185349
1,56705,2019-01-01 01:00:00,2602.0,1,ROZELLE,20001,"Western Distributor, West of Bowman Street",Western Distributor,Rozelle,5,-33.869068,151.185349
2,56705,2019-01-01 02:00:00,2312.0,1,ROZELLE,20001,"Western Distributor, West of Bowman Street",Western Distributor,Rozelle,5,-33.869068,151.185349
3,56705,2019-01-01 03:00:00,1792.0,1,ROZELLE,20001,"Western Distributor, West of Bowman Street",Western Distributor,Rozelle,5,-33.869068,151.185349
4,56705,2019-01-01 04:00:00,1428.0,1,ROZELLE,20001,"Western Distributor, West of Bowman Street",Western Distributor,Rozelle,5,-33.869068,151.185349


## 11. Create one hourly traffic feature per air-quality area

When several counters are selected for one area, the default feature is the sum across counters.

This measures the combined observed traffic volume around the area.

The table also includes:

- mean counter volume
- number of counters reporting
- standard deviation across counters

These extra columns help assess data reliability.


In [11]:
traffic_area_hourly = (
    traffic_station_hourly
    .dropna(subset=["area"])
    .groupby(["area", "DATETIME"], as_index=False)
    .agg(
        traffic_total=("traffic_count", lambda values: values.sum(min_count=1)),
        traffic_mean=("traffic_count", "mean"),
        traffic_std=("traffic_count", "std"),
        counters_reporting=("traffic_count", "count"),
    )
)

traffic_area_wide = traffic_area_hourly.pivot(
    index="DATETIME",
    columns="area",
    values="traffic_total",
)

traffic_area_wide.columns = [
    f"TRAFFIC_{area}_TOTAL"
    for area in traffic_area_wide.columns
]

traffic_area_wide = traffic_area_wide.reset_index().sort_values("DATETIME")

traffic_area_wide.to_csv(
    PROCESSED_TRAFFIC_DIR / "traffic_hourly_area_features.csv",
    index=False,
)

traffic_area_wide.head()


,DATETIME,TRAFFIC_LIVERPOOL_TOTAL,TRAFFIC_ROZELLE_TOTAL,TRAFFIC_ST_MARYS_TOTAL
0,2019-01-01 00:00:00,NaN,3129.0,NaN
1,2019-01-01 01:00:00,NaN,3617.0,NaN
2,2019-01-01 02:00:00,NaN,3131.0,NaN
3,2019-01-01 03:00:00,NaN,2433.0,NaN
4,2019-01-01 04:00:00,NaN,1874.0,NaN


## 12. Coverage and missingness report

In [12]:
expected_start = pd.Timestamp(f"{START_YEAR}-01-01 00:00:00")
expected_end = pd.Timestamp(f"{END_YEAR}-12-31 23:00:00")
expected_hours = int((expected_end - expected_start).total_seconds() / 3600) + 1

coverage_rows = []

for station_key, group in traffic_station_hourly.groupby("station_key"):
    observed = group["traffic_count"].notna().sum()

    coverage_rows.append(
        {
            "station_key": int(station_key),
            "area": group["area"].iloc[0],
            "full_name": group["full_name"].iloc[0],
            "suburb": group["suburb"].iloc[0],
            "quality_rating": group["quality_rating"].iloc[0],
            "first_datetime": group["DATETIME"].min(),
            "last_datetime": group["DATETIME"].max(),
            "observed_hours": observed,
            "expected_hours_2019_2024": expected_hours,
            "coverage_pct": observed / expected_hours * 100,
        }
    )

coverage_summary = (
    pd.DataFrame(coverage_rows)
    .sort_values(["area", "coverage_pct"], ascending=[True, False])
)

coverage_summary.to_csv(
    PROCESSED_TRAFFIC_DIR / "traffic_counter_coverage_2019_2024.csv",
    index=False,
)

coverage_summary


,station_key,area,full_name,suburb,quality_rating,first_datetime,last_datetime,observed_hours,expected_hours_2019_2024,coverage_pct
2,57176,LIVERPOOL,"Hume Highway, North of South Western Motorway",Liverpool,5,2019-01-02,2021-05-22 23:00:00,14490,52608,27.543339
3,57178,LIVERPOOL,"Elizabeth Drive, South of Reservoir Road",Liverpool,4,2019-01-18,2020-11-25 23:00:00,10245,52608,19.474224
4,15286012,ROZELLE,"Victoria Road, North of Evans Street",Rozelle,5,2019-01-01,2024-12-31 23:00:00,45700,52608,86.868917
0,56705,ROZELLE,"Western Distributor, West of Bowman Street",Rozelle,5,2019-01-01,2021-08-30 23:00:00,20349,52608,38.680429
1,56846,ROZELLE,"Victoria Road, North of Byrnes Street",Rozelle,4,2019-01-01,2019-10-21 23:00:00,6751,52608,12.832649
5,99990004,ST_MARYS,"Pages Road, North of Great Western Highway",St Marys,5,2019-01-02,2024-12-31 23:00:00,46144,52608,87.712895


## 13. Recommended station-selection rule

After reviewing `traffic_counter_coverage_2019_2024.csv`:

1. Prefer counters with a high quality rating.
2. Prefer counters with broad 2019–2024 coverage.
3. Prefer counters geographically close to the air monitor.
4. Avoid combining overlapping counters on the same road if this would double-count the same traffic stream.
5. Document the final selected station keys and justification in the report.

After changing `SELECTED_STATION_KEYS`, rerun from Section 7.


## 14. Merge with the air-quality master dataset later

Do not merge traffic into the current Scenario A/B experiment yet.

Scenario C should be added after traffic coverage is checked.

Example:

```python
scenario_c_master = air_quality_master.merge(
    traffic_area_wide,
    on="DATETIME",
    how="left",
)
```

Missing traffic values should be handled carefully:

- short gaps: causal forward fill or limited interpolation
- longer gaps: remain missing
- final unresolved model-input gaps: training-only median imputation

The traffic columns should never be used to create or fill the PM2.5 target.
